<p style="text-align: right; margin: 0; line-height: 0;"><img width="100px" height="auto" src="../../.admin/img/eth_logo.png"></p>

<h1 style="text-align: center;">02 Data Communication</h1>
<hr>


## **3. Channel Coding**

<div class="alert alert-light">

  **Name:** (add your name here)
  
  **Matriculation Number:** (add your number here)
  
</div>


### 3.1 Introduction

Source coding removes predictable structure from source data so that fewer bits need to be stored or transmitted. Before those bits are sent through a physical communication channel, channel coding adds carefully designed structure that helps the receiver detect or correct errors caused by noise, interference, or fading.

The two operations have opposite immediate goals but different purposes:

$$\text{source data} \rightarrow \text{source coding} \rightarrow \text{channel coding} \rightarrow \text{digital modulation} \rightarrow \text{channel}.$$

Source coding removes source redundancy that is not needed for reconstruction. Channel coding adds deliberate redundancy that is useful for reliable transmission.

Source redundancy is already present in the data and may be removable. Channel-coding redundancy is added intentionally and helps protect the transmitted information.



### 3.2 Code Rate

A channel encoder maps $k$ information bits to a longer codeword containing $n$ transmitted bits. The **code rate** is

$$R=\frac{\text{information bits}}{\text{transmitted bits}}=\frac{k}{n}.$$

A rate of $R=1$ means that no additional bits are transmitted. A smaller rate means that more deliberate redundancy has been added. The redundant fraction of the transmitted codeword is

$$1-R=1-\frac{k}{n}.$$

For example, if four information bits are represented by seven transmitted bits, then $R=4/7$ and the redundant fraction is $3/7$.


### 3.3 Example: Repetition Coding

A repetition code sends each information bit $m$ times. For $m=3$,

$$0\mapsto000, \qquad 1\mapsto111.$$

The receiver uses majority voting. For example, `101` is decoded as `1`, so one flipped bit can be corrected. If two or three bits flip, the majority is wrong.

Because one information bit is represented by $m$ transmitted bits, the code rate is

$$R=\frac{1}{m}.$$

For $m=3$, the rate is $R=1/3$. Two thirds of the transmitted bits are deliberate redundancy. More repetition can improve reliability when individual bit errors are sufficiently unlikely, but it also increases airtime, energy use, and delay.


### 3.3.1 When Majority Voting Fails

Assume a Binary Symmetric Channel in which each transmitted bit is flipped independently with probability $p$ and received correctly with probability $1-p$. For three repetitions, the receiver makes an error if two or three bits flip:

| Number of flips | Probability | Decoding result |
|:---:|:---:|:---|
| $0$ | $(1-p)^3$ | Correct |
| $1$ | $3p(1-p)^2$ | Correct |
| $2$ | $3p^2(1-p)$ | Incorrect |
| $3$ | $p^3$ | Incorrect |

Therefore,

$$P_{\text{error}}(3,p)=3p^2(1-p)+p^3.$$

For an odd repetition factor $m$, majority voting fails when at least $(m+1)/2$ bits flip:

$$P_{\text{error}}(m,p)=\sum_{i=(m+1)/2}^{m}\binom{m}{i}p^i(1-p)^{m-i}.$$

The binomial coefficient $\binom{m}{i}=\frac{m!}{i!(m-i)!}$ counts how many different ways exactly $i$ of the $m$ transmitted bits can flip. The factorial $m!$ means $1\cdot2\cdot3\cdots m$. For example, $3!=1\cdot2\cdot3=6$, and by definition $0!=1$. Therefore, $\binom{3}{2}=3$ because two flips can occur in positions 1 and 2, positions 1 and 3, or positions 2 and 3.


In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

def majority_error_probability(m, p):
    first_incorrect = (m + 1) // 2
    return sum(math.comb(m, i) * p**i * (1-p)**(m-i)
               for i in range(first_incorrect, m + 1))

p_values = np.linspace(0, 0.5, 201)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for m in [1, 3, 5, 7]:
    error_values = [majority_error_probability(m, p) for p in p_values]
    axes[0].plot(p_values, error_values, label=f"m = {m}")
axes[0].set_title("Error probability after majority decoding")
axes[0].set_xlabel("Channel bit-flip probability p")
axes[0].set_ylabel("Decoded bit-error probability")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

m_values = np.array([1, 3, 5, 7])
axes[1].bar(m_values.astype(str), 1 / m_values)
axes[1].set_title("More repetition lowers the code rate")
axes[1].set_xlabel("Repetition factor m")
axes[1].set_ylabel("Code rate R = 1/m")
axes[1].set_ylim(0, 1.05)
axes[1].grid(True, axis="y", alpha=0.3)

plt.tight_layout()
plt.show()


For a channel with $p<0.5$, increasing the odd repetition factor $m$ reduces the decoded bit-error probability because a larger number of bit flips is required to produce an incorrect majority. This reliability comes at a cost: the code rate falls according to $R=1/m$, so every information bit requires more transmitted bits, airtime, energy, and delay.


### 3.3.2 Structured Error-Control Codes

Repetition is easy to understand but inefficient. Practical systems add redundancy more carefully for **error detection**, **error correction**, or both. Error detection tells the receiver that the received data is probably incorrect, but it does not necessarily reveal the original data. The receiver may then discard the data or request a retransmission. Error correction uses the added structure to infer the original data directly, provided the number and pattern of errors are within the code's correction capability.

A parity bit can detect certain errors. A Cyclic Redundancy Check, or CRC, provides strong error detection. Hamming codes can correct single-bit errors. More advanced codes such as convolutional, turbo, and low-density parity-check codes provide much stronger protection for a given code rate.

The important idea is not simply to add many copies. It is to add structured patterns that a receiver can check and use during decoding.


### 3.4 Bridge to Digital Modulation

Channel coding produces a protected sequence of bits, but bits cannot travel through a wireless channel by themselves. The next notebook introduces **digital modulation**, which maps the encoded bits to physical signals that can be transmitted.


---

### 3.5 Hands-On Exercise

<div class="alert alert-block alert-info">Fill in your responses and save this notebook. See the <a href="../../03_Assignments/Submission-Guidelines.md">submission guidelines</a> for how to submit your work.</div>


**Explain the difference between source redundancy and deliberately added channel-coding redundancy. Why might a communication system remove one and later add the other?**  

(answer here)


**A repetition code uses $m=5$. What is its code rate, and what fraction of its transmitted bits is redundancy?**  

(answer here)


**Complete the repetition encoder so that each input bit is repeated $m$ times.**


In [ ]:
def repeat_encode(bits, m):
    encoded = []
    for bit in bits:
        encoded += [bit] * 0  # MODIFY HERE
    return encoded

result = repeat_encode([1, 0], 3)
expected = [1, 1, 1, 0, 0, 0]
if result == expected:
    print("Repetition encoder is correct.")
else:
    print("Not correct yet. Each bit must appear m times in the encoded list.")


**Complete the majority decoder. For each block, append 1 when more than half of its bits are 1; otherwise append 0.**


In [ ]:
def majority_decode(bits, m):
    decoded = []
    for start in range(0, len(bits), m):
        block = bits[start:start + m]
        if False:  # MODIFY HERE
            decoded.append(1)
        else:
            decoded.append(0)
    return decoded

result = majority_decode([0, 1, 1, 0, 0, 1], 3)
expected = [1, 0]
if result == expected:
    print("Majority decoder is correct.")
else:
    print("Not correct yet. Compare the number of ones with half the block length.")


**OPTIONAL: Did you understand all tasks? Which explanation or exercise should be improved?**  

(answer here)
